# 手写 AUC（Area Under ROC Curve）

## 1. AUC 的含义

**AUC = 随机取一个正样本和一个负样本，模型给正样本的预测分数高于负样本的概率。**

$$\text{AUC} = P(\hat{y}_{\text{pos}} > \hat{y}_{\text{neg}})$$

## 2. 核心概念：TPR 和 FPR

对每个阈值 $t$，把分数 $\geq t$ 的预测为正，$< t$ 的预测为负：

$$\text{TPR}(t) = \frac{TP}{TP + FN} = \frac{\text{正确识别的正样本}}{\text{正样本总数 } M}$$

$$\text{FPR}(t) = \frac{FP}{FP + TN} = \frac{\text{误判为正的负样本}}{\text{负样本总数 } N}$$

## 3. 算法步骤

1. 按分数**从高到低**排序
2. 从 (FPR=0, TPR=0) 开始，逐个降低阈值：
   - 遇到正样本 → TP += 1 → TPR 增加（向上走 $\uparrow$）
   - 遇到负样本 → FP += 1 → FPR 增加（向右走 $\rightarrow$）
3. 收集所有 (FPR, TPR) 点 → ROC 曲线
4. **梯形法则**求曲线下面积 = AUC

$$\text{AUC} = \sum_{i=1}^{n} \frac{(FPR_i - FPR_{i-1})(TPR_i + TPR_{i-1})}{2}$$

In [ ]:
import numpy as np

## 4. 代码变量与公式对应

| 代码变量 | 公式符号 | 含义 |
|----------|----------|------|
| `tp` | $TP$ | True Positive 计数 |
| `fp` | $FP$ | False Positive 计数 |
| `tp / M` | $TPR$ | 真正率（召回率） |
| `fp / N` | $FPR$ | 假正率（误报率） |
| `M` | $TP + FN$ | 正样本总数 |
| `N` | $FP + TN$ | 负样本总数 |
| `fprs[1:]-fprs[:-1]` | $\Delta FPR$ | 梯形的宽 |
| `(tprs[1:]+tprs[:-1])/2` | 平均 $TPR$ | 梯形的平均高 |

## 5. 完整实现

In [ ]:
def auc_roc(y_true, y_score):
    y_true, y_score = np.array(y_true), np.array(y_score)

    # 按分数从高到低排序
    desc = np.argsort(-y_score)
    y_sorted = y_true[desc]

    M = np.sum(y_true == 1)  # 正样本总数
    N = np.sum(y_true == 0)  # 负样本总数

    # 从 (0, 0) 开始逐步降低阈值
    tp = fp = 0
    tprs, fprs = [0.0], [0.0]

    for label in y_sorted:
        if label == 1:
            tp += 1   # 正样本 → TP+1 → 向上走
        else:
            fp += 1   # 负样本 → FP+1 → 向右走
        tprs.append(tp / M)
        fprs.append(fp / N)

    # 梯形法则求面积: 宽 × 平均高
    tprs, fprs = np.array(tprs), np.array(fprs)
    auc = np.sum((fprs[1:] - fprs[:-1]) * (tprs[1:] + tprs[:-1]) / 2)

    return auc, fprs, tprs

## 6. 测试

In [ ]:
from sklearn.metrics import roc_auc_score

y_true = np.array([1, 1, 0, 0, 1, 0, 1, 0, 0, 1])
y_score = np.array([0.9, 0.8, 0.7, 0.3, 0.6, 0.4, 0.75, 0.2, 0.1, 0.55])

auc, fpr, tpr = auc_roc(y_true, y_score)
auc_sk = roc_auc_score(y_true, y_score)

print(f"手写 AUC:   {auc:.4f}")
print(f"sklearn:    {auc_sk:.4f}")
print(f"一致: {np.isclose(auc, auc_sk)}")

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, 'b-o', ms=4, label=f'AUC={auc:.3f}')
plt.plot([0, 1], [0, 1], 'r--', label='Random')
plt.xlabel('FPR'); plt.ylabel('TPR')
plt.title('ROC Curve'); plt.legend(); plt.grid(alpha=0.3)
plt.show()

## 7. 面试追问

| 问题 | 要点 |
|------|------|
| AUC 直觉含义？ | 随机一正一负，正样本分数更高的概率 |
| TPR vs FPR？ | TPR=TP/(TP+FN) 召回率；FPR=FP/(FP+TN) 误报率 |
| ROC 怎么画？ | 按分数降序，正样本向上，负样本向右 |
| AUC vs Accuracy？ | AUC 不依赖阈值，不受样本不平衡影响 |
| AUC = 0.5？ | 模型无区分能力，等同随机 |
| AUC vs GAUC？ | GAUC 按用户分组算 AUC 再加权平均 |